In [0]:
%run ../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append("../..")

from lib.job_manager import load_config, split_config
from lib_etl.s3 import etl_input_data_validator
import lib_etl.validations_ETL as validations

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

## Data availability check

In [0]:
source_path = data_paths["source"]["awards"]
recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
etl_input_data_validator(
    "source",
    recency_lookback_duration,
    data_paths,
    [
        "awards",
    ],
    spark
)

## Load source data

In [0]:
df = spark.read.csv(source_path, header=True, sep=',', quote='"')
df.createOrReplaceTempView('df')

In [0]:
validations.validate_table(
        spark, "source", 'awards', config_validation, df, stats_etl_path
    )

## Save to delta table

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {bronze_awards}
    SELECT
        MBRSHP_SID,
        AWRD_CERT_NBR,
        AWRD_CERT_AMT,
        AWRD_CERT_ISSUE_DT,
        AWRD_CERT_EXP_DT,
        AWRD_CERT_RDMPTN_CD,
        AWRD_PROMO_ID
    FROM df
""")